# CodeConClave — real tuning on free Colab

This notebook trains two real models and saves them to your Google Drive in **`CodeConClave_Real`**:

1. **Emotion model** — reads a message and finds the feelings in it (28 emotions: joy, anger, love, sadness, fear, gratitude…). Trained on 58,000 real human messages (Google's GoEmotions dataset).
2. **An image style** — teaches an image model (Stable Diffusion 1.5) a new style from pictures (yours, or free public-domain art).

**Before you start**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Optional: to teach it **your** style, put 10–30 pictures in Google Drive folder **`CodeConClave_Real/my_images`**. With no pictures, it downloads 20 free public-domain Japanese woodblock prints (Hiroshige, Hokusai — Art Institute of Chicago, CC0) and learns that style instead.
3. Menu **Runtime → Run all**. It takes about **25–35 minutes**. Keep this tab open and in front (free Colab stops sessions that look idle).
4. **If Colab disconnects:** just press **Runtime → Run all** again. Finished steps are skipped and the picture training continues from where it stopped.

**Nothing is faked.** Every step is checked. If anything goes wrong, the notebook stops with a red **STOPPED** message and saves nothing for that step. The results (scores, before/after pictures) are saved next to the models so they can be checked.

In [ ]:
# ---- Settings (you can change these) ----
INSTANCE_PROMPT = "a picture in ccv style"   # the magic words that switch on your image style
TEST_PROMPTS = ["a cat sitting on a chair", "a city street at night",
                "a plate of chicken biryani", "a portrait of a smiling young man"]
IMAGE_STEPS = 600          # more steps = stronger style, but longer training
EMOTION_EPOCHS = 3
EMOTION_BASE = "distilbert/distilroberta-base"                 # Apache-2.0
IMAGE_BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"     # CreativeML OpenRAIL-M

In [ ]:
# ---- Step 1: check the GPU and connect Google Drive ----
import os, json, time, shutil, subprocess
from pathlib import Path

class Stopped(Exception):
    pass

def stop(msg):
    raise Stopped("STOPPED: " + msg)

def run(cmd):
    """Run a shell command, show its output live, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        stop(f"this step failed (exit code {p.returncode}). The error is just above.")

def copy_verified(src, dest):
    """Copy a folder to Drive and check every file arrived whole."""
    dest.mkdir(parents=True, exist_ok=True)
    for f in Path(src).rglob("*"):
        if f.is_file():
            t = dest / f.relative_to(src)
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(f, t)
            if t.stat().st_size != f.stat().st_size:
                stop(f"copying {f.name} to Google Drive failed")
    print("saved to", dest)

import torch
if not torch.cuda.is_available():
    stop("no GPU. Menu Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")
print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")

from google.colab import drive
drive.mount("/content/drive")
if not os.path.ismount("/content/drive"):
    stop("Google Drive did not connect. Run this cell again and allow access.")
OUT = Path("/content/drive/MyDrive/CodeConClave_Real")
(OUT / "my_images").mkdir(parents=True, exist_ok=True)
(OUT / ".write_test").write_text("ok")
if (OUT / ".write_test").read_text() != "ok":
    stop("cannot write to Google Drive (is it full?)")
(OUT / ".write_test").unlink()

REPORT = OUT / "report.json"
report = json.loads(REPORT.read_text()) if REPORT.exists() else {}
def save_report():
    REPORT.write_text(json.dumps(report, indent=2))
print("Google Drive OK ->", OUT)

In [ ]:
# ---- Step 2: train the emotion model (about 10 minutes) ----
# Already trained (report.json says done)? Then this step is skipped, so "Run all" again goes straight to the pictures.
def train_emotion():
    global report
    run("pip -q install datasets")
    import random
    import numpy as np
    from datasets import load_dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    report["emotion"] = {"status": "running", "started": time.strftime("%Y-%m-%d %H:%M")}
    save_report()
    torch.manual_seed(42); random.seed(42); np.random.seed(42)

    # GoEmotions labels, in the dataset's own order
    GO_LABELS = ["admiration", "amusement", "anger", "annoyance", "approval", "caring", "confusion",
                 "curiosity", "desire", "disappointment", "disapproval", "disgust", "embarrassment",
                 "excitement", "fear", "gratitude", "grief", "joy", "love", "nervousness", "optimism",
                 "pride", "realization", "relief", "remorse", "sadness", "surprise", "neutral"]
    ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    try:
        labels = list(ds["train"].features["labels"].feature.names)
    except Exception:
        labels = GO_LABELS
    sizes = {k: len(v) for k, v in ds.items()}
    print("examples:", sizes)
    if len(labels) != 28 or sizes.get("train", 0) < 40000:
        stop(f"the emotion dataset looks wrong ({len(labels)} labels, {sizes})")

    tok = AutoTokenizer.from_pretrained(EMOTION_BASE)
    def make_loader(split, shuffle):
        texts = list(ds[split]["text"])
        enc = tok(texts, truncation=True, max_length=64, padding="max_length", return_tensors="pt")
        y = torch.zeros(len(texts), len(labels))
        for i, ls in enumerate(list(ds[split]["labels"])):
            y[i, list(ls)] = 1.0
        data = torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"], y)
        return torch.utils.data.DataLoader(data, batch_size=32, shuffle=shuffle)
    train_dl, val_dl, test_dl = make_loader("train", True), make_loader("validation", False), make_loader("test", False)

    model = AutoModelForSequenceClassification.from_pretrained(
        EMOTION_BASE, num_labels=len(labels), problem_type="multi_label_classification",
        id2label=dict(enumerate(labels)), label2id={l: i for i, l in enumerate(labels)}).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
    total = EMOTION_EPOCHS * len(train_dl)
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    scaler = torch.amp.GradScaler("cuda")

    @torch.no_grad()
    def predict(dl):
        model.eval()
        probs, gold = [], []
        for ids, mask, y in dl:
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=ids.cuda(), attention_mask=mask.cuda()).logits
            probs.append(torch.sigmoid(logits.float()).cpu()); gold.append(y)
        model.train()
        return torch.cat(probs), torch.cat(gold)

    def f1(P, Y, thr):
        pred = (P >= thr).float()
        tp, fp, fn = (pred * Y).sum(0), (pred * (1 - Y)).sum(0), ((1 - pred) * Y).sum(0)
        micro = (2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum())).item()
        macro = (2 * tp / (2 * tp + fp + fn).clamp(min=1)).mean().item()
        return micro, macro

    # score BEFORE training, so we can prove the training changed something
    P0, Y0 = predict(test_dl)
    before_micro = f1(P0, Y0, 0.3)[0]
    print(f"score before training (micro-F1): {before_micro:.3f}")

    losses, t0 = [], time.time()
    model.train()
    for ep in range(EMOTION_EPOCHS):
        for step, (ids, mask, y) in enumerate(train_dl):
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(input_ids=ids.cuda(), attention_mask=mask.cuda(), labels=y.cuda()).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
            if step % 200 == 0:
                print(f"epoch {ep + 1}/{EMOTION_EPOCHS}  step {step}/{len(train_dl)}  loss {np.mean(losses[-200:]):.4f}", flush=True)
        P, Y = predict(val_dl)
        print(f"epoch {ep + 1} done - validation micro-F1: {f1(P, Y, 0.3)[0]:.3f}  ({(time.time() - t0) / 60:.1f} min)")

    P, Y = predict(val_dl)
    best_thr = max([0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5], key=lambda t: f1(P, Y, t)[0])
    Pt, Yt = predict(test_dl)
    micro, macro = f1(Pt, Yt, best_thr)
    print(f"\nTEST score (never seen in training): micro-F1 {micro:.3f}, macro-F1 {macro:.3f}, threshold {best_thr}")
    if micro < 0.35:
        stop(f"the emotion model did not learn well enough (micro-F1 {micro:.3f}). Nothing was saved.")

    def feel(text):
        enc = tok([text], return_tensors="pt", truncation=True, max_length=64).to("cuda")
        with torch.no_grad():
            p = torch.sigmoid(model(**enc).logits.float())[0].cpu()
        top = p.argsort(descending=True)[:3]
        return ", ".join(f"{labels[i]} {p[i]:.0%}" for i in top)
    examples = ["I hate you, you are useless", "I love this app so much!", "I'm so scared about my exam tomorrow",
                "Thank you for helping me", "My dog died today", "Wow, I didn't expect that!"]
    model.eval()
    demo = {t: feel(t) for t in examples}
    for t, r in demo.items():
        print(f"  {t!r:45} -> {r}")

    local = Path("/content/emotion_model")
    shutil.rmtree(local, ignore_errors=True)
    model.save_pretrained(local); tok.save_pretrained(local)
    info = {"labels": labels, "threshold": best_thr, "base": EMOTION_BASE,
            "dataset": "google-research-datasets/go_emotions (simplified)", "train_examples": sizes["train"],
            "epochs": EMOTION_EPOCHS, "test_micro_f1": round(micro, 4), "test_macro_f1": round(macro, 4),
            "test_micro_f1_before_training": round(before_micro, 4), "demo": demo}
    (local / "codeconclave_emotion.json").write_text(json.dumps(info, indent=2))
    copy_verified(local, OUT / "emotion_model")
    report["emotion"] = {"status": "done", "finished": time.strftime("%Y-%m-%d %H:%M"), **{k: v for k, v in info.items() if k != "labels"}}
    save_report()
    del model, opt
    torch.cuda.empty_cache()
    print("\nEMOTION MODEL DONE")

if report.get("emotion", {}).get("status") == "done":
    print("Emotion model already trained on", report["emotion"].get("finished"), "- skipping this step.")
else:
    try:
        train_emotion()
    except Exception as e:   # say so in report.json - never leave "running" behind
        report["emotion"] = {"status": "failed", "why": str(e)[:400], "at": time.strftime("%Y-%m-%d %H:%M")}
        save_report()
        raise

In [ ]:
# ---- Step 3: train your image style (about 15 minutes) ----
import numpy as np
from PIL import Image
from IPython.display import display

def train_image_style(pics, source):
    report["image"] = {"status": "running", "started": time.strftime("%Y-%m-%d %H:%M")}
    save_report()
    work = Path("/content/my_images")
    shutil.rmtree(work, ignore_errors=True); work.mkdir()
    n = 0
    for p in pics[:60]:
        try:
            im = Image.open(p).convert("RGB")
            im.thumbnail((1024, 1024))
            im.save(work / f"{n:03d}.jpg", quality=95)
            n += 1
        except Exception as e:
            print("skipping", p.name, "-", e)
    if n < 5:
        stop(f"only {n} pictures could be opened (use JPG or PNG)")
    print(f"training on {n} pictures")

    run("pip -q install diffusers peft accelerate")
    # Colab ships torchao 0.10; new peft refuses to load next to it ("only versions above 0.16.0 are supported")
    # and we do not need it, so take it out (this was the real reason the first run stopped).
    run("pip -q uninstall -y torchao")
    import diffusers
    branch = "main" if "dev" in diffusers.__version__ else f"v{diffusers.__version__}"
    src = Path("/content/diffusers_src")
    shutil.rmtree(src, ignore_errors=True)
    run(f"git clone -q --depth 1 --branch {branch} https://github.com/huggingface/diffusers {src}")
    script = src / "examples/dreambooth/train_dreambooth_lora.py"
    if not script.exists():
        stop("the official training script was not found")

    # on Google Drive, so a Colab disconnect loses nothing: "Run all" again continues from the last checkpoint
    lora_out = OUT / "image_training"
    lora_out.mkdir(parents=True, exist_ok=True)
    done_before = sorted(lora_out.glob("checkpoint-*"))
    if done_before:
        print("continuing from", done_before[-1].name)
    run(f'python "{script}" --pretrained_model_name_or_path="{IMAGE_BASE}" --instance_data_dir="{work}" '
        f'--output_dir="{lora_out}" --instance_prompt="{INSTANCE_PROMPT}" --resolution=512 '
        f'--train_batch_size=1 --gradient_accumulation_steps=1 --learning_rate=1e-4 '
        f'--lr_scheduler=constant --lr_warmup_steps=0 --max_train_steps={IMAGE_STEPS} '
        f'--checkpointing_steps=100 --checkpoints_total_limit=2 --resume_from_checkpoint=latest '
        f'--mixed_precision=fp16 --seed=42')
    weights = lora_out / "pytorch_lora_weights.safetensors"
    if not weights.exists() or weights.stat().st_size < 500_000:
        stop("training finished but no style file was made")

    # proof: same prompt + same seed, WITHOUT and WITH your style
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(IMAGE_BASE, torch_dtype=torch.float16).to("cuda")
    def gen(prompt, seed):
        g = torch.Generator("cuda").manual_seed(seed)
        return pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0]
    prompts = [f"{p}, {INSTANCE_PROMPT}" for p in TEST_PROMPTS]
    before = [gen(p, i) for i, p in enumerate(prompts)]
    pipe.load_lora_weights(str(lora_out))
    after = [gen(p, i) for i, p in enumerate(prompts)]
    diffs = [float(np.abs(np.asarray(a, dtype=np.float32) - np.asarray(b, dtype=np.float32)).mean())
             for a, b in zip(after, before)]
    print("how much your style changed each picture (0 = no change):", [round(d, 1) for d in diffs])
    if max(diffs) < 2.0:
        stop("the style had no visible effect. Try more pictures or more IMAGE_STEPS.")

    w, h = before[0].size
    grid = Image.new("RGB", (w * len(before), h * 2), "white")
    for i, (b, a) in enumerate(zip(before, after)):
        grid.paste(b, (i * w, 0)); grid.paste(a, (i * w, h))
    grid.save(lora_out / "before_top_after_bottom.png")
    print("top row = before, bottom row = after (your style)")
    display(grid.resize((w * len(before) // 2, h)))

    meta = {"base": IMAGE_BASE, "instance_prompt": INSTANCE_PROMPT, "pictures": n, "pictures_from": source, "steps": IMAGE_STEPS,
            "diffusers": diffusers.__version__, "style_change_per_picture": [round(d, 1) for d in diffs],
            "test_prompts": prompts}
    (lora_out / "codeconclave_image_style.json").write_text(json.dumps(meta, indent=2))
    for junk in lora_out.glob("checkpoint-*"):
        shutil.rmtree(junk, ignore_errors=True)
    copy_verified(lora_out, OUT / "image_style")
    shutil.rmtree(lora_out, ignore_errors=True)   # the finished style is in image_style now
    report["image"] = {"status": "done", "finished": time.strftime("%Y-%m-%d %H:%M"), **meta}
    save_report()
    del pipe
    torch.cuda.empty_cache()
    print("\nIMAGE STYLE DONE")

def fetch_public_domain_pictures(dest, want=20):
    """No pictures of your own? Download public-domain Japanese woodblock prints
    (Hiroshige, Hokusai) from the Art Institute of Chicago open-access collection (CC0, free to use)."""
    import requests
    dest.mkdir(parents=True, exist_ok=True)
    headers = {"AIC-User-Agent": "CodeConClave-tuning-notebook"}
    titles = []
    for artist in ["hiroshige", "hokusai"]:
        try:
            r = requests.get("https://api.artic.edu/api/v1/artworks/search", headers=headers, timeout=30,
                             params={"q": artist, "query[term][is_public_domain]": "true",
                                     "fields": "id,title,image_id", "limit": 40})
            r.raise_for_status()
            works = r.json().get("data", [])
        except Exception as e:
            print("could not search the museum collection:", e)
            continue
        for a in works:
            if len(titles) >= want:
                break
            if not a.get("image_id"):
                continue
            try:
                img = requests.get(f"https://www.artic.edu/iiif/2/{a['image_id']}/full/843,/0/default.jpg",
                                   headers=headers, timeout=30)
                if img.ok and img.headers.get("content-type", "").startswith("image"):
                    (dest / f"aic_{a['id']}.jpg").write_bytes(img.content)
                    titles.append(a.get("title", ""))
            except Exception as e:
                print("skipping one picture:", e)
    return titles

pics = sorted(p for p in (OUT / "my_images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))
source = "your own pictures (CodeConClave_Real/my_images)"
if len(pics) < 5:
    print(f"Found {len(pics)} pictures of yours, so I am downloading free public-domain Japanese "
          "woodblock prints (Art Institute of Chicago) to learn their style instead.")
    pub = Path("/content/public_pictures")
    shutil.rmtree(pub, ignore_errors=True)
    titles = fetch_public_domain_pictures(pub)
    pics = sorted(pub.glob("*.jpg"))
    source = f"{len(pics)} public-domain woodblock prints, Art Institute of Chicago (CC0)"
    print("downloaded:", len(pics), "pictures, e.g.", titles[:3])
if len(pics) < 5:
    report["image"] = {"status": "failed", "why": "could not get at least 5 pictures"}
    save_report()
    stop("could not get at least 5 pictures. Put 10-30 pictures in Google Drive > CodeConClave_Real > my_images and run this cell again.")
try:
    train_image_style(pics, source)
except Exception as e:   # say so in report.json - never leave "running" behind
    report["image"] = {"status": "failed", "why": str(e)[:400], "at": time.strftime("%Y-%m-%d %H:%M")}
    save_report()
    raise

In [ ]:
# ---- Step 4: summary ----
print(json.dumps(report, indent=2))
print("\nFiles saved in Google Drive > CodeConClave_Real:")
for f in sorted(OUT.rglob("*")):
    if f.is_file() and "my_images" not in f.parts:
        print(f"{f.stat().st_size / 1e6:9.1f} MB   {f.relative_to(OUT)}")
print("\nAll done. Go back to the chat and say: done")